# Closed-Loop AI G-Code Optimizer & Telemetry Agent

**Local LLM + PyTorch physics engine, validating raw CAM output against a real Haas VF-2 spindle envelope.**

```
┌─────────────┐    ┌──────────────────┐    ┌─────────────────┐    ┌──────────────────┐
│ Raw G-code  │ →  │ PyTorch physics  │ →  │ Local LLM agent │ →  │ Refined G-code   │
│ (CAM export)│    │ evaluator        │    │ (Ollama, JSON   │    │ + verified       │
│             │ ←──│ per-block power/ │ ←──│  edit contract) │ ←──│ telemetry        │
└─────────────┘    │ torque/corners   │    └─────────────────┘    └──────────────────┘
                   └──────────────────┘         closed loop, ≤6 iterations
```

**The story in one paragraph.** A CAM package exports a bracket-pocket program with mixed defaults: conservative roughing feeds on the ½″ mill and aggressive HSM slotting on the 1″ mill that was tuned for a 40 hp spindle. Our physics evaluator ingests every block, computes cutting power from specific-energy models, checks required torque against the VF-2's two-region spindle curve (constant-torque below ~1753 rpm base speed, constant-power above), and screens every corner for lateral acceleration against the axis limit. The local LLM agent reads that telemetry as JSON, proposes structured edits — feed changes, trochoidal corner fillets, entry slowdowns — and the loop repeats until the program is feasible *and* fast.

**What you'll see in this notebook:**
1. Machine envelope model (Haas VF-2 spec data)
2. A dependency-light RS-274 parser with modal state tracking
3. Vectorized PyTorch physics: MRR, spindle power/torque per block, corner kinematics
4. Baseline telemetry — the CAM export **redlines the spindle at 181%** and smashes corner limits ~14×
5. The LLM agent (Ollama) with a deterministic heuristic fallback — same edit contract
6. Closed-loop results: cycle time **6.2 min → ~3.0 min** (LLM agent) / **1.3 min** (deterministic reference), peak load ≤ 95% of envelope

> **Run it:** `py -3.13 -m venv .venv && .venv\Scripts\pip install numpy matplotlib torch --index-url https://download.pytorch.org/whl/cpu`
> then open this notebook in the repo root. If Ollama isn't running, set `AGENT = "heuristic"` in the config cell — everything else still runs.


In [ ]:
# ============================================================
# 1 · SETUP — units, machine data, agent config
# ============================================================
import json, math, re, time, urllib.request
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Arc as MArc

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
torch.manual_seed(7)
np.random.seed(7)

# --- locate repo root (notebook may be run from anywhere) ----
def _find_root():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / "data" / "sample_program.gcode").exists():
            return cand
    raise FileNotFoundError("run the notebook from the repo root")
ROOT = _find_root()

# --- where to run the physics evaluator ----------------------
# CPU is deliberate: <5k blocks -> vectorized ops take milliseconds, and the
# CPU wheel avoids CUDA DLL load risk. Ollama owns the GPUs for LLM inference.
DEVICE = torch.device("cpu")   # or "cuda" if you prefer GPU tensors

IN2MM     = 25.4        # program is G20 (inches); physics runs in mm
U_SPEC_AL = 0.8         # J/mm^3 specific cutting energy, 6061-T6
                        # (Kalpakjian & Schmid: Al alloys 0.4-1.1 W·s/mm^3)

@dataclass(frozen=True)
class MachineSpec:
    name: str
    p_max_kw: float      # continuous spindle power
    n_max_rpm: float     # max spindle speed
    t_max_nm: float      # rated torque (constant-torque region)
    a_axis_mm_s2: float  # axis acceleration limit
    f_max_mm_min: float  # max cutting feed
    rapid_mm_min: float  # rapid traverse

# Machine limits + tool library live in a JSON profile so adding another
# machine is a file drop (data/machine_profiles/<machine>.json), not a code edit.
MACHINE_PROFILE = ROOT / "data" / "machine_profiles" / "vf2.json"

def load_machine_profile(path):
    with open(path) as f:
        raw = json.load(f)
    sp, ax, fd = raw["spindle"], raw["axis"], raw["feed"]
    spec = MachineSpec(
        name=raw["name"], p_max_kw=float(sp["p_max_kw"]), n_max_rpm=float(sp["n_max_rpm"]),
        t_max_nm=float(sp["t_max_nm"]), a_axis_mm_s2=float(ax["a_axis_mm_s2"]),
        f_max_mm_min=float(fd["f_max_mm_min"]), rapid_mm_min=float(fd["rapid_mm_min"]),
    )
    tools = {int(k): dict(dia_mm=float(v["dia_mm"]), flutes=int(v["flutes"]))
             for k, v in raw.get("tools", {}).items()}
    caps  = {int(k): float(v) for k, v in raw.get("ft_cap_mm_tooth", {}).items()}
    return spec, tools, caps

# Haas VF-2 — haascnc.com spec table + Haas VF Series Service Manual (accel)
VF2, TOOLS, FT_CAP = load_machine_profile(MACHINE_PROFILE)

# --- agent config --------------------------------------------
OLLAMA_URL   = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.6:latest"    # any Ollama model works; auto-detected below
AGENT        = "auto"              # "auto" | "ollama" | "heuristic"
MAX_ITER     = 6                   # closed-loop iterations
TARGET_UTIL  = 0.95                # keep spindle <= 95% of envelope

# --- tool table + chip-load caps come from the machine profile ----
# (T1 1/2" 4-flute, T2 1" 4-flute carbide; data/machine_profiles/vf2.json)

print(f"torch {torch.__version__} on {DEVICE}")
print(f"{VF2.name}: {VF2.p_max_kw} kW | {VF2.n_max_rpm:.0f} rpm max | "
      f"{VF2.t_max_nm:.1f} N·m rated | accel {VF2.a_axis_mm_s2:.0f} mm/s²")


## 2 · The spindle envelope — two regions, one curve

Direct-drive spindles like the VF-2's are rated in **two regions**:

* **Below base speed** the motor is *torque-limited*: it can deliver its full rated torque (122 N·m here), so available power climbs linearly with rpm.
* **Above base speed** it is *power-limited*: output stays at P_max and torque falls off as 1/ω — the familiar "keep it on the pipe" region of Haas's published power/torque graphs.

Base speed is where the two meet: `n_base = 60·P_max / (2π·T_rated) ≈ 1753 rpm` for this machine. The evaluator uses the **minimum** of the two limits at every spindle speed — a conservative envelope that matches the shape of Haas's published curves.


In [ ]:
class SpindleEnvelope:
    """Two-region spindle model (constant torque -> constant power)."""
    def __init__(self, spec: MachineSpec):
        self.spec = spec
        # base speed where rated torque x omega reaches P_max
        self.n_base = 60.0 * spec.p_max_kw * 1000.0 / (2 * math.pi * spec.t_max_nm)

    def torque_avail(self, n_rpm):
        """Available torque [N·m] at spindle speed (vectorized)."""
        n = torch.as_tensor(n_rpm, dtype=torch.float64).clamp(min=1.0)
        omega = 2 * math.pi * n / 60.0
        t_power_limited = self.spec.p_max_kw * 1000.0 / omega
        return torch.minimum(torch.tensor(float(self.spec.t_max_nm), dtype=torch.float64),
                             t_power_limited)

    def power_avail(self, n_rpm):
        n = torch.as_tensor(n_rpm, dtype=torch.float64).clamp(min=1.0)
        return self.torque_avail(n) * 2 * math.pi * n / 60.0 / 1000.0

env = SpindleEnvelope(VF2)
print(f"base speed (torque -> power crossover): {env.n_base:.0f} rpm")

# --- plot the envelope ----------------------------------------
rpm = np.linspace(300, VF2.n_max_rpm, 400)
t_av = env.torque_avail(torch.tensor(rpm))
p_av = env.power_avail(torch.tensor(rpm))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(rpm, t_av.numpy(), color="#c84b31", lw=2.5, label="Available torque")
ax1.set_xlabel("Spindle speed (rpm)")
ax1.set_ylabel("Torque (N·m)", color="#c84b31")
ax1.tick_params(axis="y", labelcolor="#c84b31")
ax1.axvline(env.n_base, color="gray", ls="--", lw=1)
ax1.annotate(f"base speed ≈ {env.n_base:.0f} rpm\n(torque-limited → power-limited)",
             xy=(env.n_base, 60), xytext=(2400, 95), fontsize=9,
             arrowprops=dict(arrowstyle="->", color="gray"))
ax1.plot([2000], [VF2.t_max_nm], "o", color="#c84b31")
ax1.annotate("rated: 122 N·m @ 2000 rpm (90 ft-lbf)", xy=(2000, VF2.t_max_nm),
             xytext=(2600, 118), fontsize=9)

ax2 = ax1.twinx()
ax2.plot(rpm, p_av.numpy(), color="#457b9d", lw=2.5, ls=":", label="Available power")
ax2.set_ylabel("Power (kW)", color="#457b9d")
ax2.tick_params(axis="y", labelcolor="#457b9d")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right")
ax1.set_title(f"{VF2.name} spindle envelope — {VF2.p_max_kw} kW continuous / "
              f"{VF2.n_max_rpm:.0f} rpm max (haascnc.com spec data)")
plt.tight_layout()
plt.show()


## 3 · Ingesting raw G-code — a dependency-light RS-274 parser

Rather than pulling in a third-party parser (and its version-drift risk), the notebook ships a small modal-state parser tuned for Haas-style programs:

* **Words** are extracted with one regex; `G/M/T/S/F/X/Y/Z/I/J/R` land on structured `Block` records.
* **Modal state** is tracked across lines — motion mode (G0/G1), feed, spindle speed, active tool after `T.. M6`, and G20/G21 units.
* **Structured pass comments** like `(PASS T2 AP 0.500 AE 1.000 CLIMB)` are captured before comment-stripping. Real CAM systems emit this metadata; it tells the physics engine the *declared* axial depth (`AP`) and radial engagement (`AE`) per pass — information that is genuinely ambiguous from geometry alone (a good talking point: G-code by itself under-specifies the cut).
* Units: the program runs **G20 (inches)** like a typical Haas shop; all physics happens in mm.

The sample program `data/sample_program.gcode` is a representative CAM export for a 6061-T6 bracket pocket on the VF-2 — serpentine full-width slotting roughing with T1, then a single-pass deep slot contour with T2 at aggressive HSM defaults.


In [ ]:
WORD_RE   = re.compile(r'([A-Za-z])([-+]?\d*\.?\d+)')
PASS_RE   = re.compile(r'\(PASS\s+(\w+)\s+AP\s+([\d.]+)\s+AE\s+([\d.]+)(?:\s+(\w+))?\)', re.I)

@dataclass
class Block:
    idx: int                      # 1-based line number in the file
    text: str
    g: list = field(default_factory=list)
    m: list = field(default_factory=list)
    t = None; s = None; f = None
    x = None; y = None; z = None
    i = None; j = None; r = None

def parse_program(text: str):
    """Parse RS-274 text into structured blocks. Returns (blocks, units)."""
    blocks, units = [], "G20"
    for ln, raw in enumerate(text.splitlines(), 1):
        line = raw.strip()
        if not line or line == "%":
            continue
        b = Block(idx=ln, text=line)
        pm = PASS_RE.search(line)                      # capture before stripping
        b.pass_meta = (dict(tool=pm.group(1), ap_in=float(pm.group(2)),
                            ae_in=float(pm.group(3))) if pm else None)
        line_nc = re.sub(r'\([^)]*\)', '', line).strip()
        for m in WORD_RE.finditer(line_nc):
            k, v = m.group(1).upper(), float(m.group(2))
            if   k == 'G': b.g.append(int(v) if v.is_integer() else v)
            elif k == 'M': b.m.append(int(v))
            elif k in ('X', 'Y', 'Z', 'I', 'J'): setattr(b, k.lower(), v)
            elif k == 'R': b.r = v
            elif k == 'F': b.f = v
            elif k == 'S': b.s = v
            elif k == 'T': b.t = int(v)
        if 20 in b.g: units = "G20"
        if 21 in b.g: units = "G21"
        blocks.append(b)
    return blocks, units

gcode_text = (ROOT / "data" / "sample_program.gcode").read_text()
blocks, units = parse_program(gcode_text)
print(f"parsed {len(blocks)} blocks | units: {units}")
print("first 12 lines of the raw program:")
print("\n".join(gcode_text.splitlines()[:12]))


## 4 · From blocks to a physical trajectory

Blocks become **segments** with absolute positions (modal X/Y/Z persistence, inches → mm). Each segment is classified:

| kind | meaning | physics treatment |
|---|---|---|
| `rapid` | G0 move | time only, no load |
| `cut_xy` | G1/G2/G3 XY motion below surface | MRR = a_p·a_e·F/60 → power/torque; arcs checked for v²/R |
| `plunge` | Z-only feed move | axial chip model: MRR = tool_area · v_z |
| `idle` | air moves, M-codes | time only |

**Cutting physics (per block):**

```
MRR      = a_p · a_e · F / 60                    [mm³/s]   (a_p axial depth, a_e radial engagement)
P_cut    = u · MRR                               [W]       (u = specific cutting energy, 0.8 J/mm³ for Al)
P_total  = P_cut + P_no-load(S)                  [W]       (air-cutting/no-load grows with rpm)
T_req    = P_total / ω                           [N·m]     (ω = 2πS/60)
util     = T_req / T_avail(S)                    [-]       (>1 → spindle redline violation)
```

**Corner kinematics:** at every junction where the tool-center path changes direction by θ > 5°, the required lateral acceleration is `a_lat = v²/r_c`, with effective corner radius `r_c = δ/(1−cos(θ/2))` (δ = 0.3 mm junction-deviation allowance — what a controller can "cut off" at a sharp kink). For an unfilleted 90° corner that's r_c ≈ 1.02 mm, so corners fail above ~2370 mm/min feed on this machine. Reversals (θ > 150°, e.g. serpentine turnarounds) are checked as longitudinal stops instead: `v²/2a ≤ L_block/2`. Inserted fillet arcs of radius R replace r_c with R — which is exactly why trochoidal-style corner treatment works.


In [ ]:
@dataclass
class Seg:
    kind: str                       # rapid | cut_xy | plunge | idle
    tool: int; s_rpm: float
    x0: float; y0: float; z0: float
    x1: float; y1: float; z1: float
    f_mm_min: float = 0.0
    ap_mm: float = 0.0              # axial depth of cut [mm]
    ae_mm: float = 0.0              # radial engagement [mm] (or tool area for plunges)
    arc_R: float | None = None      # fillet radius if this segment is an arc
    g_arc: int | None = None        # 2 or 3, emitted arc direction
    src_idx: int = 0                # source line number (stable edit key)

def build_trajectory(blocks, units, tools):
    """Modal state machine -> absolute-position segments in mm."""
    sc = IN2MM if units == "G20" else 1.0
    pos = [0.0, 0.0, 0.0]
    feed, rpm, tool, pass_meta = 0.0, 0.0, None, None
    segs = []
    for b in blocks:
        if b.t is not None and 6 in b.m:          # T.. M6 -> active tool change
            tool, pass_meta = b.t, None
        if b.s is not None: rpm = b.s
        if b.f is not None: feed = b.f * sc
        if b.pass_meta: pass_meta = b.pass_meta
        g0, g1, g2, g3 = (c in b.g for c in (0, 1, 2, 3))
        if not (g0 or g1 or g2 or g3): continue
        nx = b.x * sc if b.x is not None else pos[0]
        ny = b.y * sc if b.y is not None else pos[1]
        nz = b.z * sc if b.z is not None else pos[2]
        xy_len, z_len = math.hypot(nx-pos[0], ny-pos[1]), abs(nz-pos[2])
        seg = Seg(kind="idle", tool=tool or 0, s_rpm=rpm,
                  x0=pos[0], y0=pos[1], z0=pos[2], x1=nx, y1=ny, z1=nz,
                  f_mm_min=feed, src_idx=b.idx)
        if g0:
            seg.kind = "rapid"
        elif xy_len > 1e-9 and abs(nz) < 1e-9:    # XY move at/above surface -> air
            seg.kind = "idle"
        elif xy_len > 1e-9:                       # XY cut below surface
            seg.kind = "cut_xy"
            if pass_meta and pass_meta["tool"].upper() == f"T{tool}":
                seg.ap_mm, seg.ae_mm = pass_meta["ap_in"]*IN2MM, pass_meta["ae_in"]*IN2MM
            else:                                 # fallback: conservative slotting
                seg.ap_mm, seg.ae_mm = 3.81, tools[tool]["dia_mm"] if tool in tools else 12.7
        elif z_len > 1e-9 and g1:                 # Z-only feed move (plunge/retract)
            seg.kind = "plunge"
            D = tools.get(tool, {}).get("dia_mm", 12.7)
            seg.ap_mm, seg.ae_mm = 1.0, math.pi*D*D/4   # MRR = area * v_z
        if g2 or g3:
            seg.arc_R = b.r * sc if b.r is not None else None
            seg.g_arc = 2 if g2 else 3
        pos = [nx, ny, nz]
        segs.append(seg)
    return segs

def seg_len(s):
    """Path length of a segment (arcs from chord + radius)."""
    if s.kind == "cut_xy":
        if s.arc_R:
            chord = math.hypot(s.x1-s.x0, s.y1-s.y0)
            phi = 2 * math.asin(min(1.0, chord / (2*s.arc_R)))
            return s.arc_R * phi
        return math.hypot(s.x1-s.x0, s.y1-s.y0)
    if s.kind == "plunge":
        return abs(s.z1 - s.z0)
    return 0.0

def no_load_kw(n_rpm):
    """No-load / air-cutting power [kW] — typical for a 30 hp direct-drive spindle."""
    return torch.as_tensor(n_rpm, dtype=torch.float64) * 0.000125 + 1.0

JUNC_DEV = 0.3   # mm junction-deviation allowance at sharp corners

def evaluate(segs, spec=VF2, env=None, u_spec=U_SPEC_AL):
    """Vectorized physics evaluation -> telemetry dict (torch tensors)."""
    env = env or SpindleEnvelope(spec)
    cuts = [s for s in segs if s.kind in ("cut_xy", "plunge")]
    f  = torch.tensor([s.f_mm_min for s in cuts], dtype=torch.float64)
    ap = torch.tensor([s.ap_mm for s in cuts], dtype=torch.float64)
    ae = torch.tensor([s.ae_mm for s in cuts], dtype=torch.float64)
    S  = torch.tensor([s.s_rpm for s in cuts], dtype=torch.float64)
    L  = torch.tensor([seg_len(s) for s in cuts], dtype=torch.float64)

    mrr     = ap * ae * f / 60.0                 # mm³/s
    p_cut   = u_spec * mrr                       # W
    p_total = p_cut + no_load_kw(S) * 1000.0     # W
    omega   = 2 * math.pi * S / 60.0
    t_req   = p_total / omega                    # N·m
    util    = t_req / env.torque_avail(S)        # vs available at that rpm

    t_cut_min   = float((L / f).sum()) if len(cuts) else 0.0
    rapids      = [s for s in segs if s.kind == "rapid"]
    t_rapid_min = sum(math.hypot(s.x1-s.x0, s.y1-s.y0, s.z1-s.z0) / spec.rapid_mm_min / 60
                      for s in rapids)

    # ---- corner analysis at junctions between consecutive XY cuts ----------
    corners, violations = [], []
    xy = [i for i, s in enumerate(segs) if s.kind == "cut_xy"]
    pairs = [(k, k+1) for k in range(len(xy)-1)]
    # closed loops: within each contiguous tool/Z group, a last segment that ends
    # where the first one began adds a wrap-around (closing-corner) junction
    g0 = 0
    while g0 < len(xy):
        g1 = g0 + 1
        while (g1 < len(xy) and segs[xy[g1]].tool == segs[xy[g0]].tool
               and abs(segs[xy[g1]].z0 - segs[xy[g0]].z0) < 1e-6):
            g1 += 1
        if g1 - g0 >= 3:
            a_, b_ = segs[xy[g1-1]], segs[xy[g0]]
            if math.hypot(b_.x0-a_.x1, b_.y0-a_.y1) < 1e-6:
                pairs.append((g1-1, g0))
        g0 = g1

    for k, k2 in pairs:
        a_, b_ = segs[xy[k]], segs[xy[k2]]
        if a_.tool != b_.tool or abs(a_.z1-b_.z0) > 1e-6: continue
        if math.hypot(b_.x0-a_.x1, b_.y0-a_.y1) > 1e-6: continue   # rapid in between
        u1 = np.array([a_.x1-a_.x0, a_.y1-a_.y0]); l1 = float(np.linalg.norm(u1))
        u2 = np.array([b_.x1-b_.x0, b_.y1-b_.y0]); l2 = float(np.linalg.norm(u2))
        if l1 < 1e-9 or l2 < 1e-9: continue
        cos_t = max(-1.0, min(1.0, float(np.dot(u1, u2) / (l1*l2))))
        theta = math.degrees(math.acos(cos_t))
        if theta <= 5.0: continue
        v = a_.f_mm_min / 60.0                    # approach speed [mm/s]
        rec = dict(seg_a=xy[k], seg_b=xy[k2], src_key=a_.src_idx, theta=theta)
        if theta > 150.0:                         # reversal -> longitudinal stop check
            d_stop = v*v / (2*spec.a_axis_mm_s2)
            ok = d_stop <= 0.5 * min(l1, l2)
            rec.update(kind="reversal", a_req=v*v/max(d_stop, 1e-9), limit=spec.a_axis_mm_s2, ok=ok)
        else:                                     # true corner -> lateral acceleration
            r_c = JUNC_DEV / (1 - math.cos(math.radians(theta)/2))
            a_lat = v*v / r_c
            ok = a_lat <= spec.a_axis_mm_s2
            rec.update(kind="corner", r_c=r_c, a_req=a_lat, limit=spec.a_axis_mm_s2, ok=ok)
        corners.append(rec)
        if not ok:
            violations.append((xy[k], "corner_lateral" if theta <= 150 else "corner_reversal"))

    for i, (s, u) in enumerate(zip(cuts, util)):  # spindle load violations
        if u.item() > 1.0:
            violations.append((segs.index(s), "spindle_load", f"util {u.item()*100:.0f}%"))
    for i, s in enumerate(segs):                 # lateral accel along inserted arcs
        if s.kind == "cut_xy" and s.arc_R:
            a_arc = (s.f_mm_min/60.0)**2 / s.arc_R
            if a_arc > spec.a_axis_mm_s2:
                violations.append((i, "arc_lateral", f"a={a_arc:.0f} mm/s² on R{s.arc_R:.1f}"))

    vol = float((ap * ae * L).sum())             # removed volume [mm³]
    return dict(n=len(cuts), mrr=mrr, p_total=p_total, t_req=t_req, util=util,
                t_cut_min=t_cut_min, t_rapid_min=t_rapid_min, corners=corners,
                violations=violations, vol_mm3=vol,
                cuts_idx=[segs.index(s) for s in cuts])

segs = build_trajectory(blocks, units, TOOLS)
print("segment mix:", dict(Counter(s.kind for s in segs)))


In [ ]:
def arc_points(p1, p2, R, ccw):
    """Sample points along a circular arc from p1 to p2 with radius R."""
    import numpy as _np
    p1, p2 = _np.asarray(p1, float), _np.asarray(p2, float)
    mid, d = (p1+p2)/2.0, p2-p1
    chord = float(_np.linalg.norm(d))
    if chord < 1e-9: return [tuple(p1)]
    h = math.sqrt(max(R*R - (chord/2)**2, 0.0))
    n = _np.array([-d[1], d[0]]) / chord          # left normal of chord direction
    pts = []
    for sgn in (+1, -1):
        C = mid + sgn * h * n
        a1, a2 = math.atan2(p1[1]-C[1], p1[0]-C[0]), math.atan2(p2[1]-C[1], p2[0]-C[0])
        sweep = (a2-a1) % (2*math.pi) if ccw else (a1-a2) % (2*math.pi)
        if 0 < sweep <= math.pi + 1e-6:           # minor arc in the right sense
            ts = _np.linspace(0, 1, max(int(sweep/0.05), 8))
            pts = [tuple(C + R*_np.array([math.cos(a1+sweep*t), math.sin(a1+sweep*t)])) for t in ts]
            break
    return pts or [tuple(p1), tuple(p2)]

def draw_toolpath(ax, segs, highlight=None, title=""):
    """Plot the XY toolpath; highlight = set of segment indices to mark red."""
    highlight = highlight or set()
    for i, s in enumerate(segs):
        if s.kind not in ("cut_xy", "rapid"): continue
        col = "#d62828" if i in highlight else ("#457b9d" if s.tool == 1 else "#2a9d8f")
        lw  = 2.0 if i in highlight else (1.2 if s.kind == "cut_xy" else 0.6)
        ls  = "-" if s.kind == "cut_xy" else ":"
        if s.arc_R:
            ccw = (s.g_arc or 3) == 3
            pts = arc_points((s.x0, s.y0), (s.x1, s.y1), s.arc_R, ccw)
            ax.plot([p[0] for p in pts], [p[1] for p in pts], color=col, lw=lw, ls=ls, zorder=3 if i in highlight else 2)
        else:
            ax.plot([s.x0, s.x1], [s.y0, s.y1], color=col, lw=lw, ls=ls, zorder=3 if i in highlight else 2)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
    ax.set_xlabel("X (mm)"); ax.set_ylabel("Y (mm)")

# ---------------- baseline evaluation ----------------
tel = evaluate(segs)
total_min = tel["t_cut_min"] + tel["t_rapid_min"]
print(f"BASELINE  cut={tel['t_cut_min']:.2f} min  rapid={tel['t_rapid_min']:.3f} min  total={total_min:.2f} min")
print(f"volume ≈ {tel['vol_mm3']:,.0f} mm³   avg MRR = {tel['vol_mm3']/max(tel['t_cut_min'],1e-9):,.0f} mm³/min")
print(f"peak spindle utilization: {float(tel['util'].max())*100:.1f}% of envelope")
print("violations:", len(tel["violations"]), dict(Counter(v[1] for v in tel["violations"])))

# ---- plots -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))

hl = {v[0] for v in tel["violations"]}
draw_toolpath(axes[0], segs, highlight=hl, title="Baseline toolpath — red = violating blocks")
for c in [c for c in tel["corners"] if not c["ok"]]:
    a_ = segs[c["seg_a"]]
    axes[0].plot([a_.x1], [a_.y1], "kx", ms=9, mew=2)
axes[0].legend(handles=[plt.Line2D([], [], color="#457b9d", lw=2, label="T1 ½″ roughing"),
                        plt.Line2D([], [], color="#2a9d8f", lw=2, label="T2 1″ deep slotting"),
                        plt.Line2D([], [], color="#d62828", lw=2, label="violating segment"),
                        plt.Line2D([], [], marker="x", ls="", color="k", ms=9, mew=2, label="corner violation")],
               fontsize=8, loc="upper left")

# power trace vs time
cuts = [segs[i] for i in tel["cuts_idx"]]
t = np.concatenate([[0.0], np.cumsum([seg_len(s)/s.f_mm_min*60 for s in cuts])])
p_kw = (tel["p_total"].numpy()/1000.0)
p_ext = np.append(p_kw, p_kw[-1])          # step() wants equal-length x/y
axes[1].step(t, p_ext, where="post", color="#2a9d8f", lw=1.5, label="spindle power demand")
axes[1].axhline(VF2.p_max_kw, color="#c84b31", ls="--", lw=2, label=f"envelope {VF2.p_max_kw} kW")
axes[1].fill_between(t, VF2.p_max_kw, max(p_kw.max(), VF2.p_max_kw)*1.05, where=p_ext > VF2.p_max_kw,
                     color="#d62828", alpha=0.35, label="redline region")
axes[1].set_xlabel("cycle time (min)"); axes[1].set_ylabel("spindle power (kW)")
axes[1].set_title("Baseline spindle power trace — T2 slotting redlines the VF-2")
axes[1].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()


## 5 · The LLM agent — telemetry in, structured edits out

The agent is **not** a code generator and it doesn't do gradient descent. It's a decision layer with a strict contract:

* **Input:** a compact JSON *telemetry summary* from the physics evaluator — per-tool utilization, chip-load caps, every violating block, and (crucially) *all* sharp junctions of each tool so corners can be filleted proactively when feed is raised.
* **Output:** exactly one JSON object of edits drawn from a small vocabulary:

```json
{"edits": [
  {"type": "set_feed",   "tool": 2,              "f_mm_min": 4540},
  {"type": "set_feed",   "segs": [81, 84],       "f_mm_min": 2310},
  {"type": "corner_arc", "src_key": 81,          "r_mm": 12.7}
]}
```

* **Backends:** `OllamaAgent` (local model over HTTP — no cloud, no API key) and a deterministic `HeuristicAgent` that implements the same policy in closed form (power-envelope feed solve + chip-load caps + fillet-or-slowdown corner rule). `AGENT="auto"` probes Ollama with a 5-second timeout and falls back to the heuristic, so **the notebook never hard-fails** — important when demoing on job-fair Wi-Fi.
* **Robustness:** JSON is parsed defensively (fences stripped, schema validated, values clamped); each closed-loop iteration keeps the *best verified feasible state* - and if no iteration reaches full feasibility within budget, the loop returns the closest-to-feasible state it saw. A lucky-but-overreaching LLM move can't wreck the final program either way.


In [ ]:
import copy

def apply_edits(segs, edits):
    """Apply agent edits -> new segment list.

    set_feed  : tool-wide (XY cuts only; plunges keep their own feed) or per-line via segs=[src_idx...]
    corner_arc: fillet the junction entered by the segment with src_key=src_idx
    """
    segs = [copy.copy(s) for s in segs]

    # 1) feeds first (segment indices stay stable)
    for e in edits:
        if e["type"] != "set_feed": continue
        F, targets = float(e["f_mm_min"]), set(e.get("segs", []))
        for i, s in enumerate(segs):
            if s.kind not in ("cut_xy", "plunge"): continue
            if e.get("tool") is not None and s.tool == e["tool"] and s.kind == "cut_xy":
                segs[i].f_mm_min = F
            elif targets and s.src_idx in targets:
                segs[i].f_mm_min = F

    # 2) corner arcs — fillet geometry at the junction
    for e in edits:
        if e["type"] != "corner_arc": continue
        R, key = float(e["r_mm"]), e.get("src_key")
        idx_a = next((i for i, s in enumerate(segs) if s.src_idx == key and s.kind == "cut_xy"), None)
        if idx_a is None or idx_a + 1 >= len(segs): continue
        a_, b_ = segs[idx_a], segs[idx_a+1]
        if not (a_.kind == "cut_xy" and b_.kind == "cut_xy" and a_.tool == b_.tool
                and math.hypot(b_.x0-a_.x1, b_.y0-a_.y1) < 1e-6): continue
        if b_.arc_R is not None: continue          # already filleted
        u1 = np.array([a_.x1-a_.x0, a_.y1-a_.y0]); l1 = float(np.linalg.norm(u1))
        u2 = np.array([b_.x1-b_.x0, b_.y1-b_.y0]); l2 = float(np.linalg.norm(u2))
        if l1 < 1e-9 or l2 < 1e-9: continue
        cos_t = max(-1.0, min(1.0, float(np.dot(u1, u2) / (l1*l2))))
        theta = math.acos(cos_t)
        d = R * math.tan(theta/2)                   # corner -> tangent-point distance
        if d >= min(l1, l2) - 1.0: continue         # fillet must fit in remaining length
        u1n, u2n = u1/l1, u2/l2
        C = np.array([a_.x1, a_.y1])
        P1, P2 = C - u1n*d, C + u2n*d               # tangent points
        cross_z = float(u1[0]*u2[1] - u1[1]*u2[0])
        g_arc = 3 if cross_z > 0 else 2             # CCW vs CW turn
        a_.x1, a_.y1 = float(P1[0]), float(P1[1])   # truncate incoming segment
        b_.x0, b_.y0 = float(P2[0]), float(P2[1])   # start outgoing past the fillet
        arc = Seg(kind="cut_xy", tool=a_.tool, s_rpm=a_.s_rpm,
                  x0=float(P1[0]), y0=float(P1[1]), z0=a_.z1,
                  x1=float(P2[0]), y1=float(P2[1]), z1=b_.z0,
                  f_mm_min=a_.f_mm_min, ap_mm=a_.ap_mm, ae_mm=a_.ae_mm,
                  arc_R=R, g_arc=g_arc, src_idx=a_.src_idx)
        segs.insert(idx_a + 1, arc)
    return segs

def emit_gcode(segs, units="G20"):
    """Regenerate a Haas-style program from the edited trajectory (inches)."""
    inv = 1.0/IN2MM if units == "G20" else 1.0
    out, last_tool = [], None
    a = out.append
    a("%")
    a("O1042 (BRACKET POCKET - 6061-T6) [AGENT-OPTIMIZED]")
    a("( MACHINE : HAAS VF-2 / CT40 DIRECT DRIVE )")
    a("( OPTIMIZED BY CLOSED-LOOP LLM AGENT + PHYSICS EVALUATOR )")
    a(f"G90 G17 {units} G40 G80")
    a("G54")
    for s in segs:
        if s.kind == "idle": continue
        if s.tool != last_tool and s.tool in TOOLS:
            D_in = TOOLS[s.tool]["dia_mm"] * inv
            a("")
            a(f"T{s.tool} M6 ({D_in:.3f} IN 4FL CARBIDE END MILL)")
            a(f"S{int(s.s_rpm)} M3")
            last_tool = s.tool
        if s.kind == "rapid":
            parts = []
            if abs(s.x1-s.x0) > 1e-9: parts.append(f"X{s.x1*inv:.3f}")
            if abs(s.y1-s.y0) > 1e-9: parts.append(f"Y{s.y1*inv:.3f}")
            if abs(s.z1-s.z0) > 1e-9: parts.append(f"Z{s.z1*inv:.3f}")
            a("G0 " + " ".join(parts))
        elif s.kind == "plunge":
            a(f"G1 Z{s.z1*inv:.3f} F{s.f_mm_min*inv:.2f}")
        elif s.kind == "cut_xy" and s.arc_R:
            g = s.g_arc or 3
            a(f"G{g} X{s.x1*inv:.3f} Y{s.y1*inv:.3f} R{s.arc_R*inv:.3f} F{s.f_mm_min*inv:.2f}")
        elif s.kind == "cut_xy":
            parts = []
            if abs(s.x1-s.x0) > 1e-9: parts.append(f"X{s.x1*inv:.3f}")
            if abs(s.y1-s.y0) > 1e-9: parts.append(f"Y{s.y1*inv:.3f}")
            a("G1 " + " ".join(parts) + f" F{s.f_mm_min*inv:.2f}")
    a("")
    a("M5")
    a("M30")
    a("%")
    return "\n".join(out) + "\n"


In [ ]:
def telemetry_summary(segs, tel):
    """Compact JSON-able view of the physics state for the agent."""
    tools = {}
    cuts_all = [s for s in segs if s.kind in ("cut_xy", "plunge")]
    util_by_seg = {id(s): float(u) for s, u in zip(cuts_all, tel["util"])}
    for t in sorted({s.tool for s in segs if s.kind in ("cut_xy", "plunge")}):
        ts = [i for i, s in enumerate(segs) if s.tool == t and s.kind == "cut_xy"]
        peak = max(util_by_seg[id(s)] for s in segs if s.tool == t and s.kind in ("cut_xy", "plunge"))
        sharp = [c for c in tel["corners"]
                 if c.get("kind") == "corner" and c["theta"] >= 60.0 and segs[c["seg_a"]].tool == t]
        tools[str(t)] = dict(
            dia_mm=TOOLS[t]["dia_mm"], flutes=TOOLS[t]["flutes"], s_rpm=int(segs[ts[0]].s_rpm),
            feeds_mm_min=sorted({round(s.f_mm_min, 1) for i, s in enumerate(segs) if s.tool == t and s.kind == "cut_xy"}),
            peak_util_pct=round(peak*100, 1), chip_load_cap_mm_tooth=FT_CAP.get(t),
            sharp_corners=dict(count=len(sharp),
                               src_keys=[segs[c["seg_a"]].src_idx for c in sharp],
                               theta_deg=round(sharp[0]["theta"], 1) if sharp else None,
                               violating=sum(1 for c in sharp if not c["ok"])))
    viol, seen = [], set()
    for v in tel["violations"]:
        if v[1] == "spindle_load":
            s = segs[v[0]]
            p_kw = round(float(tel["p_total"][cuts_all.index(s)])/1000.0, 1)
            key = (s.tool, p_kw)
            if key in seen: continue
            seen.add(key)
            viol.append(dict(type="spindle_load", tool=s.tool,
                             src_lines=[x.src_idx for x in segs if x.tool == s.tool and x.kind == "cut_xy"],
                             p_total_kw=p_kw, envelope_kw=VF2.p_max_kw))
        elif v[1] == "corner_lateral":
            c = next(c for c in tel["corners"] if not c["ok"] and c.get("kind") == "corner" and c["seg_a"] == v[0])
            a_ = segs[c["seg_a"]]
            viol.append(dict(type="corner_lateral", tool=a_.tool, src_key=a_.src_idx,
                             theta_deg=round(c["theta"], 1),
                             a_req_mm_s2=round(c["a_req"]), limit_mm_s2=int(VF2.a_axis_mm_s2)))
    return dict(machine=dict(name=VF2.name, p_max_kw=VF2.p_max_kw, t_rated_nm=round(VF2.t_max_nm, 1),
                             a_axis_mm_s2=int(VF2.a_axis_mm_s2), f_max_mm_min=int(VF2.f_max_mm_min)),
                tools=tools, violations=viol,
                cycle_time_min=round(tel["t_cut_min"] + tel["t_rapid_min"], 3),
                avg_mrr_mm3_min=round(tel["vol_mm3"]/max(tel["t_cut_min"], 1e-9)))

SYSTEM_PROMPT = """You are CAM-AGENT, an autonomous CNC optimization agent in a closed loop with a physics evaluator that checks every block of the G-code against real machine limits.

MACHINE (Haas VF-2): spindle 22.4 kW continuous / 8100 rpm max; rated torque 122 N·m below ~1753 rpm base speed, power-limited above it. Axis acceleration limit 1524 mm/s². Max cutting feed 16510 mm/min.

TOOLS: T1 D=12.7mm 4-flute S=8000rpm chip-load cap 0.10 mm/tooth. T2 D=25.4mm 4-flute S=6000rpm chip-load cap 0.25 mm/tooth.

WHAT THE EVALUATOR CHECKS:
- Spindle load: P = u*MRR + no_load(S), MRR = ap*ae*F/60 mm³/s, u=0.8 J/mm³ (aluminum). Violation when required torque exceeds available at that rpm.
- Corners: sharp direction changes need lateral accel v²/r_c <= 1524 mm/s²; unfilleted 90° corners have r_c≈1.02mm, so they fail above ~2370 mm/min feed. Fix with a corner arc (fillet) or a slowdown into the corner.
- Arcs: lateral accel v²/R along any inserted arc must also stay <= 1524 mm/s².

EDIT VOCABULARY — reply with exactly ONE JSON object, no prose, no markdown fences:
{"edits":[ ... ]} where each edit is one of:
- {"type":"set_feed","tool":<int>,"f_mm_min":<float>}  (all XY cuts for that tool; plunges keep their own feed)
- {"type":"set_feed","segs":[<src line ints>],"f_mm_min":<float>}  (specific source lines only)
- {"type":"corner_arc","src_key":<int src line of the segment ENTERING the corner>,"r_mm":<float>}

RULES:
1. Bring every tool's peak spindle utilization to <= 95% of envelope.
2. Fix EVERY violating corner. Prefer arcs with r up to half the tool diameter; use slowdowns (set_feed on segs) only where an arc cannot fit or at the loop-closing junction.
3. Never exceed chip-load caps, max feed, or the machine acceleration limit. Treat peak utilization >= 94% as AT-LIMIT: do not raise that tool's feed further.
4. If a tool is underutilized (< 40%) with no violations, speed it up toward its binding limit (chip load or power). When you raise a tool's feed, fillet ALL of its sharp corners (listed in tools.<t>.sharp_corners.src_keys) in the same batch — unfilleted 90° corners fail above ~2370 mm/min.
5. Use src line numbers exactly as given in telemetry."""

def _ollama_chat(messages, timeout=240):
    body = json.dumps(dict(model=OLLAMA_MODEL, stream=False, think=False, format="json",
                           options=dict(num_predict=6000, temperature=0.2),
                           messages=messages)).encode()
    last_err = None
    for _attempt in range(2):                  # one retry: truncated responses happen
        req = urllib.request.Request(OLLAMA_URL + "/api/chat", data=body,
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode())
        except (ValueError, OSError) as ex:     # JSONDecodeError / URLError / timeout
            last_err = ex
    raise RuntimeError(f"Ollama unreadable after 2 tries ({last_err.__class__.__name__})")

def _parse_edits(content):
    c = content.strip()
    if c.startswith("```"):
        c = c.split("\n", 1)[1].rsplit("```", 1)[0]
    try:
        edits = json.loads(c).get("edits", [])
    except (ValueError, AttributeError):        # malformed payload -> no-op iteration
        return [], []
    valid, dropped = [], []
    for e in edits:
        t = e.get("type")
        if t == "set_feed" and ("tool" in e or "segs" in e) and "f_mm_min" in e:
            e["f_mm_min"] = max(50.0, min(float(e["f_mm_min"]), VF2.f_max_mm_min))
            valid.append(e)
        elif t == "corner_arc" and "src_key" in e and "r_mm" in e:
            e["r_mm"] = max(1.0, min(float(e["r_mm"]), 30.0))
            valid.append(e)
        else:
            dropped.append(e)
    return valid, dropped

class OllamaAgent:
    """Local LLM backend (Ollama HTTP API)."""
    name = "ollama"
    def propose(self, segs, tel):
        summary = telemetry_summary(segs, tel)
        try:
            resp = _ollama_chat([dict(role="system", content=SYSTEM_PROMPT),
                                 dict(role="user", content=json.dumps(summary))])
            edits, dropped = _parse_edits(resp["message"]["content"])
        except Exception as ex:                  # keep the loop alive (demo-safe)
            return [], dict(model=OLLAMA_MODEL, error=f"{ex.__class__.__name__}: {ex}")
        return edits, dict(model=OLLAMA_MODEL, dropped=len(dropped), summary=summary)

class HeuristicAgent:
    """Deterministic policy backend — same edit vocabulary, closed-form math."""
    name = "heuristic"
    def propose(self, segs, tel):
        env_ = SpindleEnvelope(VF2)
        edits = []
        cuts_all = [s for s in segs if s.kind in ("cut_xy", "plunge")]
        util_by_seg = {id(s): float(u) for s, u in zip(cuts_all, tel["util"])}
        # --- per-tool feed decisions -------------------------------------
        for t in sorted({s.tool for s in segs if s.kind == "cut_xy"}):
            t_segs = [i for i, s in enumerate(segs) if s.tool == t and s.kind == "cut_xy"]
            peak = max(util_by_seg[id(s)] for s in segs if s.tool == t and s.kind == "cut_xy")
            s0 = segs[t_segs[0]]
            omega = 2*math.pi*s0.s_rpm/60.0
            p0 = float(no_load_kw(torch.tensor(float(s0.s_rpm))))*1000.0
            f_target = (TARGET_UTIL*float(env_.torque_avail(s0.s_rpm))*omega - p0) * 60.0 / (U_SPEC_AL*s0.ap_mm*s0.ae_mm)
            chip_cap = FT_CAP.get(t, 0.25)*TOOLS[t]["flutes"]*s0.s_rpm
            if peak > TARGET_UTIL:
                new_f = min(f_target, chip_cap, VF2.f_max_mm_min)
            elif peak < 0.40:                       # exploit underutilized tools
                new_f = min(chip_cap, f_target, VF2.f_max_mm_min)
            else:
                continue
            if abs(new_f - s0.f_mm_min)/max(s0.f_mm_min, 1e-9) > 0.02 and new_f > 50:
                edits.append(dict(type="set_feed", tool=t, f_mm_min=round(new_f, 1)))
        # --- corners still violating after feed changes -------------------
        sim = apply_edits(segs, [e for e in edits if e["type"] == "set_feed"])
        tel2 = evaluate(sim)
        for c in tel2["corners"]:
            if c["ok"] or c.get("kind") != "corner": continue
            a_, b_ = sim[c["seg_a"]], sim[c["seg_b"]]
            R_tool = TOOLS[a_.tool]["dia_mm"]/2.0
            l1 = math.hypot(a_.x1-a_.x0, a_.y1-a_.y0)
            l2 = math.hypot(b_.x1-b_.x0, b_.y1-b_.y0)
            R = min(R_tool, 0.44*min(l1, l2)/math.tan(math.radians(c["theta"])/2))
            arc_ok = False
            if R >= 3.0:
                nxt = sim[c["seg_a"]+1] if c["seg_a"]+1 < len(sim) else None
                is_closing = not (nxt is not None and nxt.kind == "cut_xy"
                                  and math.hypot(nxt.x0-a_.x1, nxt.y0-a_.y1) < 1e-6)
                if not is_closing:                  # closing junctions can't take an inline arc
                    edits.append(dict(type="corner_arc", src_key=a_.src_idx, r_mm=round(R, 2)))
                    arc_ok = True
            if not arc_ok:                          # decelerate both sides of the junction
                v_corner = math.sqrt(0.95*VF2.a_axis_mm_s2 * c["r_c"])
                edits.append(dict(type="set_feed", segs=[a_.src_idx, b_.src_idx],
                                  f_mm_min=round(60.0*v_corner, 1)))
        return edits, dict(model=f"heuristic (closed-form)")

def select_agent():
    if AGENT == "heuristic":
        return HeuristicAgent()
    if AGENT == "ollama":
        return OllamaAgent()
    try:                                            # auto: probe Ollama with a short timeout
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=5) as r:
            models = [m["name"] for m in json.loads(r.read().decode()).get("models", [])]
        if not any(m.startswith(OLLAMA_MODEL.split(":")[0]) for m in models):
            print(f"note: {OLLAMA_MODEL} not found; available: {models}")
        return OllamaAgent()
    except Exception as ex:
        print(f"Ollama unreachable ({ex.__class__.__name__}) -> falling back to heuristic agent")
        return HeuristicAgent()

agent = select_agent()
print("active agent:", agent.name)


In [ ]:
def run_closed_loop(segs, tel, agent, max_iter=MAX_ITER):
    """Iterate: evaluate -> propose edits -> apply -> re-evaluate.

    Keeps the best verified feasible state (0 violations) so an overreaching
    move can't wreck the final program. If no iteration reaches full feasibility
    within budget, returns the closest-to-feasible state seen (fewest violations,
    ties broken by cycle time) - never worse than the input. Returns
    (segs_final, tel_final, log).
    """
    log, best = [], None
    near = ((len(tel["violations"]), tel["t_cut_min"] + tel["t_rapid_min"]), segs, tel)
    for it in range(1, max_iter + 1):
        t0 = time.time()
        edits, meta = agent.propose(segs, tel)
        dt = time.time() - t0
        n_feed = sum(1 for e in edits if e["type"] == "set_feed")
        n_arc  = sum(1 for e in edits if e["type"] == "corner_arc")
        row = dict(iter=it, agent=agent.name, dt_s=round(dt, 1), feeds=n_feed, arcs=n_arc,
                   v_before=len(tel["violations"]))
        log.append(row)
        print(f"iter {it}: [{agent.name} {dt:.1f}s] feeds={n_feed} arcs={n_arc} "
              f"violations {len(tel['violations'])}", end="")
        if not edits:
            row["v_after"] = len(tel["violations"])
            print("  -> no edits, converged")
            break
        segs_new = apply_edits(segs, edits)
        tel2 = evaluate(segs_new)
        row["v_after"] = len(tel2["violations"])
        row["time_min"] = round(tel2["t_cut_min"] + tel2["t_rapid_min"], 3)
        print(f" -> {len(tel2['violations'])}   time {(tel['t_cut_min']+tel['t_rapid_min']):.2f}"
              f" -> {(tel2['t_cut_min']+tel2['t_rapid_min']):.2f} min")
        if not tel2["violations"]:
            cand = (tel2["t_cut_min"] + tel2["t_rapid_min"], segs_new, tel2)
            if best is None or cand[0] < best[0]:
                best = cand
        else:
            key = (len(tel2["violations"]), tel2["t_cut_min"] + tel2["t_rapid_min"])
            if key < near[0]:
                near = (key, segs_new, tel2)
        segs, tel = segs_new, tel2
    if best is not None and (tel["violations"] or best[1] is not segs):
        print("best-so-far feasible state selected as final program")
        return best[1], best[2], log
    if near[0][0] < len(tel["violations"]):
        print(f"no fully feasible state within {max_iter} iterations; "
              f"returning closest-to-feasible ({near[0][0]} violations)")
        return near[1], near[2], log
    return segs, tel, log

segs_opt, tel_opt, loop_log = run_closed_loop(segs, tel, agent)


## 6 · Results — from redline to verified

The loop stops when the agent has nothing left to improve (or `MAX_ITER` is hit). The final program is **verified by re-running the same physics evaluator** — not trusted on faith. Below: before/after toolpaths, power traces, and the headline numbers. The optimized G-code is written to `data/optimized_program.gcode`.


In [ ]:
# ---------------- before / after comparison ----------------
base_min = tel["t_cut_min"] + tel["t_rapid_min"]
opt_min  = tel_opt["t_cut_min"] + tel_opt["t_rapid_min"]
mrr_base = tel["vol_mm3"] / max(tel["t_cut_min"], 1e-9)
mrr_opt  = tel_opt["vol_mm3"] / max(tel_opt["t_cut_min"], 1e-9)

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))

# toolpaths side by side
draw_toolpath(axes[0][0], segs, title=f"Baseline — {base_min:.2f} min, "
             f"{len(tel['violations'])} violations (infeasible)")
hl_opt = {v[0] for v in tel_opt["violations"]}
draw_toolpath(axes[0][1], segs_opt, highlight=hl_opt,
              title=f"Agent-optimized — {opt_min:.2f} min, "
                    f"{len(tel_opt['violations'])} violations")

# power traces side by side
def _power_trace(ax, segs_, tel_):
    cuts = [segs_[i] for i in tel_["cuts_idx"]]
    t = np.concatenate([[0.0], np.cumsum([seg_len(s)/max(s.f_mm_min,1e-9)*60 for s in cuts])])
    p_kw = tel_["p_total"].numpy()/1000.0
    ax.step(t, np.append(p_kw, p_kw[-1]), where="post", color="#2a9d8f", lw=1.4)
    ax.axhline(VF2.p_max_kw, color="#c84b31", ls="--", lw=1.6)
    ax.set_xlabel("cycle time (min)"); ax.set_ylabel("spindle power (kW)")

_power_trace(axes[1][0], segs, tel);   axes[1][0].set_title(f"Baseline power — peak {float(tel['p_total'].max())/1000:.1f} kW")
_power_trace(axes[1][1], segs_opt, tel_opt)
axes[1][1].set_title(f"Optimized power — peak {float(tel_opt['p_total'].max())/1000:.1f} kW (envelope {VF2.p_max_kw} kW)")

plt.suptitle("Closed-loop optimization on the Haas VF-2", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

# ---------------- headline metrics ----------------
print("=" * 68)
print(f"{'metric':<34}{'baseline':>15}{'optimized':>15}")
print("-" * 68)
print(f"{'cycle time (min)':<34}{base_min:>15.2f}{opt_min:>15.2f}")
print(f"{'sustained MRR (mm³/min)':<34}{mrr_base:>15,.0f}{mrr_opt:>15,.0f}")
print(f"{'peak spindle utilization':<34}{float(tel['util'].max())*100:>14.1f}%{float(tel_opt['util'].max())*100:>14.1f}%")
print(f"{'violations (infeasible blocks)':<34}{len(tel['violations']):>15d}{len(tel_opt['violations']):>15d}")
print(f"{'cycle time speedup':<34}{'—':>15}{base_min/max(opt_min,1e-9):>14.2f}x")
print("=" * 68)

# per-tool final feeds
for t in (1, 2):
    feeds = sorted({round(s.f_mm_min) for s in segs_opt if s.tool == t and s.kind == "cut_xy"})
    print(f"tool {t}: final XY-cut feeds = {feeds} mm/min")

# ---------------- write the optimized program ----------------
opt_gcode = emit_gcode(segs_opt)
out_path = ROOT / "data" / "optimized_program.gcode"
out_path.write_text(opt_gcode)
print(f"\nwrote {out_path.relative_to(ROOT)} ({len(opt_gcode.splitlines())} lines)")

# ---------------- reference: deterministic heuristic (if LLM ran) --------
if agent.name == "ollama":
    print("\n--- reference run with the deterministic HeuristicAgent ---")
    segs_ref, tel_ref, _ = run_closed_loop(segs, tel, HeuristicAgent(), max_iter=MAX_ITER)
    ref_min = tel_ref["t_cut_min"] + tel_ref["t_rapid_min"]
    print(f"heuristic: {ref_min:.2f} min | peak util {float(tel_ref['util'].max())*100:.1f}% "
          f"| violations {len(tel_ref['violations'])}")


## 7 · Findings & limitations

**What the loop found (this run):**

1. **The CAM export was infeasible.** T2's single-pass deep slotting at F≈9000 mm/min / S6000 demanded ~40 kW — **181% of the VF-2's 22.4 kW envelope** — and its four unfilleted 90° corners needed ~22,000 mm/s² of lateral acceleration against a 1524 mm/s² axis limit (~14×).
2. **The fix was rebalancing, not just slowing down.** The agent cut T2 to the power-envelope feed (~45% of its original), filleted every sharp corner (R up to tool radius — trochoidal-style), slowed only the loop-closing junction where an inline arc can't fit, *and* raised T1's conservative roughing feed ~5× to its chip-load limit. Net in this recorded run: **cycle time dropped ~2× and peak load landed at 94% of envelope; one corner block still trips the lateral-acceleration limit when the iteration budget runs out, so the loop returns the closest-to-feasible state it saw.**
3. **LLM vs. heuristic:** the deterministic policy converges in 2 iterations to the same optimum every run; the local LLM clears most violations within 2–4 iterations, with run-to-run variance (temperature sampling) — this recorded run is the honest case where it stops one block short of fully feasible and the closest-to-feasible fallback takes over. Both share one edit contract and one evaluator — swap backends without touching physics.

**Walkthrough (~4 min):**
1. Show the raw G-code + the redlined power trace ("this is what CAM gave us").
2. Run the closed-loop cell live; narrate each iteration as it prints (feeds → arcs → verify).
3. Open `data/optimized_program.gcode` — point out the inserted `G2/G3 … R…` fillets and the slowed entry corner.
4. Land the number: same part, ~2× faster with the LLM agent (1.3 min fully feasible with the deterministic reference), and the spindle stays inside its envelope.

**Honest limitations:**
* Specific cutting energy is a constant (0.8 J/mm³ for 6061-T6); real u varies with chip load, tool wear and coolant. The model is *directionally* right and conservative at the envelope boundary.
* No-load power is a linear-in-rpm approximation; Haas publishes per-machine curves that could be dropped straight into `no_load_kw()`.
* Corner analysis uses a junction-deviation model (δ = 0.3 mm) rather than full servo look-ahead — it catches the failures that matter for this program class, not every following-error scenario.
* Engagement widths come from CAM pass comments (`AP`/`AE`); without them the engine falls back to conservative slotting assumptions.
* The LLM is a 23B local model at temperature 0.2 — great for structured edits, not a substitute for the evaluator that actually verifies every block.

**References:** Haas VF-2 spec tables (haascnc.com); Haas VF Series Service Manual (axis acceleration ≈ 60 in/s²); Kalpakjian & Schmid, *Manufacturing Engineering and Technology* (specific cutting energy: Al alloys 0.4–1.1 W·s/mm³); Ollama local inference API; PyTorch vectorized evaluation on CPU.
